In [1]:
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()
data_path = os.getenv("DATA_PATH")

In [4]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import col


In [ ]:
df = pd.read_csv(data_path)

df.replace("?", np.nan, inplace=True)

for col in ['workclass', 'occupation', 'native.country']:
    df[col].fillna(df[col].mode()[0], inplace=True)

df.drop_duplicates(inplace=True)


In [6]:
print("----- Pandas EDA -----")

# Shape
print("Shape:", df.shape)

# Data types
print("\nData Types:\n", df.dtypes)

# Summary statistics
print("\nSummary Statistics:\n", df.describe())

# Income distribution
print("\nIncome Distribution:\n", df['income'].value_counts())

# Correlation (numerical)
print("\nCorrelation Matrix:\n", df.corr(numeric_only=True))

----- Pandas EDA -----
Shape: (32537, 15)

Data Types:
 age                int64
workclass         object
fnlwgt             int64
education         object
education.num      int64
marital.status    object
occupation        object
relationship      object
race              object
sex               object
capital.gain       int64
capital.loss       int64
hours.per.week     int64
native.country    object
income            object
dtype: object

Summary Statistics:
                 age        fnlwgt  education.num  capital.gain  capital.loss  \
count  32537.000000  3.253700e+04   32537.000000  32537.000000  32537.000000   
mean      38.585549  1.897808e+05      10.081815   1078.443741     87.368227   
std       13.637984  1.055565e+05       2.571633   7387.957424    403.101833   
min       17.000000  1.228500e+04       1.000000      0.000000      0.000000   
25%       28.000000  1.178270e+05       9.000000      0.000000      0.000000   
50%       37.000000  1.783560e+05      10.000000     

In [9]:
spark = SparkSession.builder.appName("AdultEDA").getOrCreate()

spark_df = spark.read.csv(data_path, header=True, inferSchema=True)

# Rename columns (important!)
for col_name in spark_df.columns:
    spark_df = spark_df.withColumnRenamed(col_name, col_name.replace(".", "_"))

spark_df.printSchema()
spark_df.describe().show()

# Group by income
spark_df.groupBy("income").count().show()

# Stop Spark session
spark.stop()

root
 |-- age: integer (nullable = true)
 |-- workclass: string (nullable = true)
 |-- fnlwgt: integer (nullable = true)
 |-- education: string (nullable = true)
 |-- education_num: integer (nullable = true)
 |-- marital_status: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- relationship: string (nullable = true)
 |-- race: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- capital_gain: integer (nullable = true)
 |-- capital_loss: integer (nullable = true)
 |-- hours_per_week: integer (nullable = true)
 |-- native_country: string (nullable = true)
 |-- income: string (nullable = true)

+-------+------------------+-----------+------------------+------------+-----------------+--------------+----------------+------------+------------------+------+------------------+-----------------+------------------+--------------+------+
|summary|               age|  workclass|            fnlwgt|   education|    education_num|marital_status|      occupation|relat

🔥 Difference Between Pandas and Spark (Viva Gold)
Pandas                                Spark
Works in-memory	                      Distributed computing
Suitable for small-medium data	      Suitable for big data
Single machine	                      Cluster-based
Faster for small datasets	          Scalable for large datasets

🎯 Important Viva Points
Why use Spark if dataset is small?

To demonstrate distributed data processing capability.

What is Spark?

An open-source distributed computing system for big data processing.

What is SparkSession?

Entry point for working with Spark DataFrames.